# Agno Framework — Introduction

**Agno** is the evolution of phidata — a lightweight, high-performance agent framework.
Key advantages over phidata:
- Natively model-agnostic (OpenAI, Gemini, Anthropic, Groq, local models)
- Built-in multi-agent `Team` primitive
- Agno OS: a beautiful UI for running and debugging agents
- ~10,000 agents/sec throughput (memory-efficient)

Install: `pip install agno openai duckduckgo-search`

In [1]:
# ! pip install agno openai duckduckgo-search yfinance python-dotenv

### Environment Setup
We use OpenRouter to access multiple model providers through a single API key.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Clear any conflicting env vars
for var in ['OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_API_BASE']:
    if os.getenv(var):
        print(f"⚠️  Removing conflicting {var}")
        del os.environ[var]

# Route through OpenRouter
os.environ["OPENAI_API_KEY"] = os.getenv("OPEN_ROUTER_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"

print("✅ Environment configured for OpenRouter")

✅ Environment configured for OpenRouter


### Your First Agno Agent
The `Agent` class is the core primitive. It wraps a model with instructions, tools, and memory.

In [3]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

agent = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    description="You are a helpful assistant with a knack for clear explanations.",
    markdown=True,
)

agent.print_response("Explain agentic AI in 3 sentences.", stream=True)

Output()

### Structured Output with Agno
Use `output_schema=` with a Pydantic model to get structured JSON responses.

> **Agno 2.x note**: the old `response_model=` parameter was renamed to `output_schema=`.

In [4]:
from typing import List
from pydantic import BaseModel, Field
from agno.agent import Agent
from agno.models.openai import OpenAIChat

class AgentPattern(BaseModel):
    name: str = Field(..., description="Pattern name")
    category: str = Field(..., description="Category: single-agent, multi-agent, or iterative")
    use_case: str = Field(..., description="Primary use case in one sentence")
    complexity: str = Field(..., description="Low / Medium / High")
    key_benefits: List[str] = Field(..., description="Top 3 benefits")

pattern_agent = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    description="You are an AI architecture expert.",
    output_schema=AgentPattern,   # Agno 2.x — was response_model= in older versions
    structured_outputs=True,
)

result = pattern_agent.run("Describe the ReAct (Reason and Act) agentic pattern")
print(result.content)

name='ReAct (Reason and Act)' category='single-agent' use_case='Enables language models to perform complex tasks by interweaving reasoning (Chain-of-Thought) and acting (tool use) steps.' complexity='Medium' key_benefits=['Improved task performance through dynamic planning and execution.', 'Enhanced interpretability by showing reasoning traces.', 'Ability to interact with external environments and tools.']


### Memory & History

Agno agents remember past turns using `SqliteDb` + `add_history_to_context`.

**Agno 2.x API** (renamed from phidata):

| Old | New |
|-----|-----|
| `storage=SqliteStorage(...)` | `db=SqliteDb(...)` |
| `add_history_to_messages=True` | `add_history_to_context=True` |
| `num_history_responses=N` | `num_history_runs=N` |

**Rules**:
- Use `db_file="chat.db"` (file-based), NOT `sqlite:///:memory:` — each `:memory:` connection is a separate empty DB
- Use `agent.run()` to read back a response with memory; `print_response()` is fine for display-only turns
- Do NOT re-run the cell that creates `Agent(...)` between turns — that starts a new session

In [5]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.db.sqlite import SqliteDb

chat_agent = Agent(
    model=OpenAIChat(id="google/gemini-2.5-flash"),
    description="You are a helpful assistant.",
    add_history_to_context=True,
    num_history_runs=5,
    db=SqliteDb(db_file="chat.db"),
)

# Turn 1 — store a fact
chat_agent.print_response("My name is John and I am studying agentic AI.")



Output()

In [6]:
# Turn 2 — verify memory: agent should recall name and topic
result = chat_agent.run("What was my name and what am I studying?")
print(result.content)

Your name is **John**, and you are studying **agentic AI**.


In [7]:
import sqlite3

conn = sqlite3.connect("chat.db")
cursor = conn.cursor()

cursor.execute("SELECT agent_id, runs FROM agno_sessions ORDER BY created_at DESC LIMIT 1")
agent_id, runs_raw = cursor.fetchone()

print(f"Agent ID: {agent_id}")
print(f"Runs raw type: {type(runs_raw)}")
print(f"Runs raw value: {repr(runs_raw[:500])}")  # first 500 chars raw

conn.close()

Agent ID: brave-pasteur-183bc656
Runs raw type: <class 'str'>
Runs raw value: '"[{\\"run_id\\": \\"8539a907-19fb-4af0-9c4e-72383ddcc960\\", \\"agent_id\\": \\"brave-pasteur-183bc656\\", \\"session_id\\": \\"d5fa92bd-ddc6-447b-b4df-f7394eb5df08\\", \\"content\\": \\"Nice to meet you, John! That\'s a fascinating field to be studying. Agentic AI is really at the cutting edge of artificial intelligence, focusing on creating AI systems that can operate autonomously, make decisions, and achieve goals in complex environments.\\\\n\\\\nTo help me understand what might be most useful for you, could yo'
